In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import pandas as pd
root = Path.cwd()
if root.name in ["train", "notebooks"]:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from src.data.dataset import create_dataloaders
from src.models.lstm import LSTMLanguageModel
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


Cấu hình thí nghiệm


In [ ]:
CONTEXT_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 50                      # Tổng số epoch mong muốn
PATIENCE = 5
LEARNING_RATE = 0.001
GRAD_CLIP = 1.0
RESUME_TRAINING = False          # True để tiếp tục từ last_checkpoint.pt
RESET_PATIENCE_ON_RESUME = True
USE_TENSORBOARD = True
MODEL_NAME = "lstm"
EXPERIMENT_NAME = f"word_c{CONTEXT_LENGTH}"

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


Tải dữ liệu và khởi tạo LSTM


In [7]:
train_loader, valid_loader, test_loader, vocab_size = create_dataloaders(
    context_length=CONTEXT_LENGTH, batch_size=BATCH_SIZE
)
model = LSTMLanguageModel(
    vocab_size=vocab_size, embedding_dim=256, hidden_dim=256, num_layers=2, dropout=0.2
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Vocabulary size:", vocab_size)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


Vocabulary size: 10000
Model parameters: 6,180,624


Kiểm tra Forward Pass

In [8]:
X, Y = next(iter(train_loader))

X = X.to(device)

model.eval()

with torch.no_grad():
    logits = model(X)

print("Input shape:", X.shape)
print("Target shape:", Y.shape)
print("Output shape:", logits.shape)

assert logits.shape == (
    X.shape[0],
    X.shape[1],
    vocab_size
)

print("LSTM hoạt động thành công!")

Input shape: torch.Size([16, 128])
Target shape: torch.Size([16, 128])
Output shape: torch.Size([16, 128, 10000])
LSTM hoạt động thành công!


Huấn luyện LSTM trực tiếp theo từng epoch


In [13]:
# ==========================================================
# TRAIN / VALIDATION / CHECKPOINT - vòng lặp PyTorch trực tiếp
# ==========================================================
import math
import time
from tqdm.auto import tqdm

CHECKPOINT_DIR = root / "checkpoints" / MODEL_NAME / EXPERIMENT_NAME
RESULTS_DIR = root / "results" / MODEL_NAME / EXPERIMENT_NAME
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = CHECKPOINT_DIR / "best_model.pt"       # Chỉ state_dict, phục vụ test/inference
LAST_PATH = CHECKPOINT_DIR / "last_checkpoint.pt"  # Đầy đủ trạng thái, phục vụ resume
HISTORY_PATH = RESULTS_DIR / "history.csv"

writer = None
if USE_TENSORBOARD:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir=str(RESULTS_DIR / "tensorboard"))

start_epoch = 0
best_valid_loss = float("inf")
patience_counter = 0
history = []
previous_training_time = 0.0

if RESUME_TRAINING:
    if not LAST_PATH.exists():
        raise FileNotFoundError(f"Không tìm thấy checkpoint để tiếp tục: {LAST_PATH}")
    checkpoint = torch.load(LAST_PATH, map_location=device, weights_only=False)
    if checkpoint.get("context_length") != CONTEXT_LENGTH or checkpoint.get("vocab_size") != vocab_size:
        raise ValueError("Checkpoint có context_length hoặc vocab_size khác cấu hình hiện tại")
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = int(checkpoint["epoch"])
    best_valid_loss = float(checkpoint["best_valid_loss"])
    patience_counter = int(checkpoint.get("patience_counter", 0))
    history = checkpoint.get("history", [])
    previous_training_time = float(checkpoint.get("training_time_seconds", 0.0))
    if RESET_PATIENCE_ON_RESUME:
        patience_counter = 0
    # Khôi phục RNG của CPU
    cpu_rng = checkpoint.get("torch_rng_state")

    if cpu_rng is not None:
        cpu_rng = torch.as_tensor(
            cpu_rng,
            dtype=torch.uint8,
            device="cpu"
        )
        torch.set_rng_state(cpu_rng)

    # Khôi phục RNG của GPU
    cuda_rng = checkpoint.get("cuda_rng_state_all")

    if device.type == "cuda" and cuda_rng is not None:

        cuda_rng = [
            torch.as_tensor(
                state,
                dtype=torch.uint8,
                device="cpu"
            )
            for state in cuda_rng
        ]

        torch.cuda.set_rng_state_all(cuda_rng)

    print("Đã khôi phục RNG state!")
        
    print(f"Đã resume từ epoch {start_epoch}; tiếp tục đến epoch {EPOCHS}.")

if device.type == "cuda":
    torch.cuda.synchronize()
start_time = time.time()

for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # ======================= TRAINING PHASE =======================
    model.train()
    train_loss_sum = 0.0
    train_tokens = 0
    train_bar = tqdm(train_loader, desc=f"Train {epoch + 1}/{EPOCHS}", leave=False)

    for batch_idx, (X, Y) in enumerate(train_bar):
        X, Y = X.to(device), Y.to(device)

        # Forward -> Loss
        logits = model(X)
        loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))

        # Backward -> Gradient Clipping -> Update
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        n_tokens = Y.numel()
        train_loss_sum += loss.item() * n_tokens
        train_tokens += n_tokens
        train_bar.set_postfix(loss=f"{train_loss_sum / train_tokens:.4f}")

        if writer is not None:
            writer.add_scalar("Train/BatchLoss", loss.item(), epoch * len(train_loader) + batch_idx)

    train_loss = train_loss_sum / train_tokens
    train_ppl = math.exp(train_loss)

    # ======================= VALIDATION PHASE =====================
    model.eval()
    valid_loss_sum = 0.0
    valid_tokens = 0
    with torch.no_grad():
        val_bar = tqdm(valid_loader, desc=f"Valid {epoch + 1}/{EPOCHS}", leave=False)
        for X, Y in val_bar:
            X, Y = X.to(device), Y.to(device)
            logits = model(X)
            loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
            n_tokens = Y.numel()
            valid_loss_sum += loss.item() * n_tokens
            valid_tokens += n_tokens
            val_bar.set_postfix(loss=f"{valid_loss_sum / valid_tokens:.4f}")

    valid_loss = valid_loss_sum / valid_tokens
    valid_ppl = math.exp(valid_loss)
    print(f"Train Loss: {train_loss:.4f} | Train PPL: {train_ppl:.2f}")
    print(f"Valid Loss: {valid_loss:.4f} | Valid PPL: {valid_ppl:.2f}")

    history.append({
        "epoch": epoch + 1, "context_length": CONTEXT_LENGTH, "batch_size": BATCH_SIZE,
        "train_loss": train_loss, "train_ppl": train_ppl,
        "valid_loss": valid_loss, "valid_ppl": valid_ppl,
    })
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if writer is not None:
        writer.add_scalar("Train/Loss", train_loss, epoch + 1)
        writer.add_scalar("Train/PPL", train_ppl, epoch + 1)
        writer.add_scalar("Valid/Loss", valid_loss, epoch + 1)
        writer.add_scalar("Valid/PPL", valid_ppl, epoch + 1)
        writer.flush()

    # ====================== SAVE BEST MODEL =======================
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        patience_counter = 0
        torch.save(model.state_dict(), BEST_PATH)
        print("Đã lưu best_model.pt")
    else:
        patience_counter += 1
        print(f"Không cải thiện: {patience_counter}/{PATIENCE}")

    # ====================== SAVE LAST CHECKPOINT ==================
    if device.type == "cuda":
        torch.cuda.synchronize()
    total_training_time = previous_training_time + (time.time() - start_time)
    checkpoint = {
        "epoch": epoch + 1,  # Số epoch đã hoàn thành
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_valid_loss": best_valid_loss,
        "patience_counter": patience_counter,
        "history": history,
        "training_time_seconds": total_training_time,
        "context_length": CONTEXT_LENGTH,
        "batch_size": BATCH_SIZE,
        "vocab_size": vocab_size,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
    }
    torch.save(checkpoint, LAST_PATH)
    print("Đã lưu last_checkpoint.pt")

    # ======================== EARLY STOPPING ======================
    if patience_counter >= PATIENCE:
        print("Early Stopping!")
        break

if device.type == "cuda":
    torch.cuda.synchronize()
training_time = previous_training_time + (time.time() - start_time)
if writer is not None:
    writer.close()
print(f"Training time (total): {training_time / 60:.2f} phút")
print("Best model:", BEST_PATH)
print("Last checkpoint:", LAST_PATH)


Đã khôi phục RNG state!
Đã resume từ epoch 1; tiếp tục đến epoch 50.

Epoch 2/50


Train 2/50:   0%|          | 0/1042 [00:00<?, ?it/s]

KeyboardInterrupt: 

Đánh giá trên tập Test


In [ ]:
# Đánh giá trên TEST SET bằng BEST MODEL
if not BEST_PATH.exists():
    raise FileNotFoundError(f"Chưa có best model: {BEST_PATH}")
model.load_state_dict(torch.load(BEST_PATH, map_location=device, weights_only=True))
model.eval()
test_loss_sum, test_tokens = 0.0, 0
with torch.no_grad():
    for X, Y in tqdm(test_loader, desc="Test", leave=False):
        X, Y = X.to(device), Y.to(device)
        logits = model(X)
        loss = criterion(logits.reshape(-1, logits.size(-1)), Y.reshape(-1))
        n_tokens = Y.numel()
        test_loss_sum += loss.item() * n_tokens
        test_tokens += n_tokens

test_loss = test_loss_sum / test_tokens
test_ppl = math.exp(test_loss)
print(f"Test Loss: {test_loss:.4f} | Test PPL: {test_ppl:.2f}")


Tổng hợp kết quả


In [ ]:
summary = pd.DataFrame([{
    "model": MODEL_NAME,
    "experiment": EXPERIMENT_NAME,
    "parameters": sum(p.numel() for p in model.parameters()),
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "epochs_trained": len(history),
    "best_valid_loss": best_valid_loss,
    "best_valid_ppl": math.exp(best_valid_loss),
    "test_loss": test_loss,
    "test_ppl": test_ppl,
    "training_time_seconds": training_time,
}])
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
display(summary)


In [ ]:
print("Train samples:", len(train_loader.dataset))
print("Train batches:", len(train_loader))
print("Batch size:", train_loader.batch_size)
print("Tokens per sample:", train_loader.dataset.context_length)
print("Total train tokens:", len(train_loader.dataset.tokens))

Train samples: 33335
Train batches: 1042
Batch size: 32
Tokens per sample: 64
Total train tokens: 2133462
